In [1]:
import pandas as pd
import numpy as np

In [ ]:
data_path='../raw_datasets/'
eth_regs=pd.read_csv(f'{data_path}eth_domains_registrations.csv').drop(['registrant_address','registration_date'], axis=1)
eth_regs=eth_regs.rename(columns={'labelhash_hex':'labelhash'})

In [3]:
eth_to_usd=pd.read_csv(f'{data_path}ETHUSDT_1d.csv')
eth_to_usd['open_time_ms'] = pd.to_datetime(eth_to_usd['open_time_ms'], unit='ms')
eth_price=eth_to_usd.sort_values(by='open_time_ms', ascending=False).iloc[0]['open_price']
eth_regs['eth_reg_price']=np.round(eth_regs['price']*eth_price,2)

Переходим к историям транзакций

In [4]:
eth_sales_base=pd.read_csv(f'{data_path}eth_domains_sales_base.csv').drop(['tx_hash','unix_timestamp'], axis=1)
eth_sales_wrap=pd.read_csv(f'{data_path}eth_domains_sales_wrapped.csv').drop(['tx_hash','unix_timestamp'], axis=1)


У врап версии есть значения price_usd = "nil" 

In [5]:
eth_sales_wrap['price_usd'].value_counts()

price_usd
<nil>                  64
0.21477150000000003    21
0.21519400000000002    20
0.2148805              16
0.09892691             15
                       ..
19.07388                1
0.642309                1
84.33959999999999       1
1815.9066000000003      1
434.77719               1
Name: count, Length: 6213, dtype: int64

In [6]:
eth_sales_wrap=eth_sales_wrap[eth_sales_wrap['price_usd']!='<nil>']
eth_sales_wrap['price_usd']=eth_sales_wrap['price_usd'].astype(float)
eth_sales_base['namehash'] = eth_sales_base['labelhash'].map(eth_regs.set_index('labelhash')['namehash'])


In [7]:
eth_sales_base['namehash'].isna().sum()

np.int64(16)

16 пропущенных значений в namehash, эти строки нам не нужны т.к это те домены которых нет в списке регистраций

Объединяем списки транзакций

In [8]:
eth_sales=pd.concat([eth_sales_wrap,
eth_sales_base.drop('labelhash', axis=1)], ignore_index=True
)

In [9]:
eth_sales_agg= (
    eth_sales.groupby('namehash', as_index=False).agg(
        eth_sales_count=('price_usd', 'count'),
        eth_sales_volume_usd=('price_usd', 'sum'),
        eth_last_sale_price_usd=('price_usd', 'last'),
    )
)

In [10]:
eth_res=eth_regs.merge(eth_sales_agg, on='namehash', how='left')


Откидываем eth из domain_name

In [11]:
eth_res['domain_name']=eth_res['domain_name'].apply( lambda x : x.split(".")[0] )

То же самое с sol 

In [12]:
sol_reg=pd.read_csv(f'{data_path}sol_domains_registrations.csv').drop(['tx_signature','bidder_key','unix_timestamp'], axis=1)
sol_sales=pd.read_csv(f'{data_path}sol_domains_sales.csv').drop(['tx_signature','bidder_key','unix_timestamp'], axis=1)

In [13]:
sol_sales_agg = (
    sol_sales.groupby('domain_key', as_index=False)
    .agg(
        sol_sales_count=('usd_price', 'count'),
        sol_sales_volume_usd=('usd_price', 'sum'),
        sol_last_sale_price_usd=('usd_price', 'last'),
    )
)


In [14]:
sol_res=sol_reg.merge(sol_sales_agg, on='domain_key', how='left')

In [15]:
sol_res

,domain_name,domain_key,usd_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd
0,chase,BXU5YtHFxMZhSsFiM8xDSUeFAqv3rB6TuUdReKMNq8S2,20.010000,NaN,NaN,NaN
1,first,GPgDU7EpeaZMFYaMmTMeSEF9dHakuKVm4rivNXLPgvGN,45.000000,NaN,NaN,NaN
2,ethereum,52WzzPis6oVdr7G21d3k1n82t43xsLbU5BPyBfhq2mSa,85.000000,1.0,2.313413e+05,231341.28
3,raydium,5DXrekrp1WzxkqNX9m7Hm72yhUcXcPAYF4kJrN7sHQc9,180.000000,43.0,1.305256e+06,5000.00
4,sam,EdKTQpwxykgrqCAygeHgK33YJLAAicYdMZUXvg6WQBCq,110.000000,9.0,2.662477e+05,25000.00
...,...,...,...,...,...,...
460186,olmec,DEziar1HiFW8SH1TyKijj1XuLntycjkkWhHUfU4JvRDP,20.397518,NaN,NaN,NaN
460187,cambi,4ReV3dqyjhsiq7Cmjr8NxaLhW3pwK82yUCyh2dPZZnLQ,20.328514,NaN,NaN,NaN
460188,cambicasa,2VCKQ9zqBk9yphiipukAsRjUiojdTEUZSKnnr7YJHwz8,20.328514,NaN,NaN,NaN
460189,louisxv,7jqwD7Wu4sCJFAfDBLghe6Xv32y9qgwC5QtV78mhpEV7,20.328514,NaN,NaN,NaN


In [16]:
eth_res.drop(['namehash','labelhash'], axis=1,inplace=True)
sol_res.drop('domain_key', axis=1,inplace=True)
result_df=eth_res.merge(sol_res, on='domain_name', how='left')

In [17]:
len(result_df[result_df.notnull().all(axis=1)])

8565

In [18]:
result_df['domain_name'].duplicated().sum()

np.int64(0)

In [ ]:
result_df.to_csv('../datasets/secondary_prices.csv', index=False)